In [1]:
import torch
from transformers import MambaForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "state-spaces/mamba-1.4b-hf"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16  # or float16
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token  # Mamba doesn't have a pad token; use eos

model = MambaForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation=None  # Mamba doesn't use attention, so no need to set
)

# Prepare model for k-bit training (necessary for gradient checkpointing etc.)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

/home/gaanfok/anaconda3/envs/mamba/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 35544.95it/s]
The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.
Loading weights: 100%|██████████| 482/482 [00:01<00:00, 371.21it/s, Materializing param=backbone.norm_f.weight]                  


In [2]:
import torch.nn as nn

for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        print(name)

backbone.layers.0.mixer.in_proj
backbone.layers.0.mixer.x_proj
backbone.layers.0.mixer.dt_proj
backbone.layers.0.mixer.out_proj
backbone.layers.1.mixer.in_proj
backbone.layers.1.mixer.x_proj
backbone.layers.1.mixer.dt_proj
backbone.layers.1.mixer.out_proj
backbone.layers.2.mixer.in_proj
backbone.layers.2.mixer.x_proj
backbone.layers.2.mixer.dt_proj
backbone.layers.2.mixer.out_proj
backbone.layers.3.mixer.in_proj
backbone.layers.3.mixer.x_proj
backbone.layers.3.mixer.dt_proj
backbone.layers.3.mixer.out_proj
backbone.layers.4.mixer.in_proj
backbone.layers.4.mixer.x_proj
backbone.layers.4.mixer.dt_proj
backbone.layers.4.mixer.out_proj
backbone.layers.5.mixer.in_proj
backbone.layers.5.mixer.x_proj
backbone.layers.5.mixer.dt_proj
backbone.layers.5.mixer.out_proj
backbone.layers.6.mixer.in_proj
backbone.layers.6.mixer.x_proj
backbone.layers.6.mixer.dt_proj
backbone.layers.6.mixer.out_proj
backbone.layers.7.mixer.in_proj
backbone.layers.7.mixer.x_proj
backbone.layers.7.mixer.dt_proj
backbone.

In [3]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["in_proj", "x_proj", "dt_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 14,376,960 || all params: 1,386,555,392 || trainable%: 1.0369


In [4]:
from datasets import load_dataset

openbookqa = load_dataset("openbookqa")  # has train, validation, test
arc_easy = load_dataset("ai2_arc", "ARC-Easy")
arc_challenge = load_dataset("ai2_arc", "ARC-Challenge")

In [5]:
def format_openbookqa_train(example):
    question = example["question_stem"]
    choices = example["choices"]["text"]
    answer = example["answerKey"]  # letter like "A", "B", etc.
    # Get the correct choice text
    correct_choice = choices[ord(answer) - ord('A')]  # works for A,B,C,D
    # Build text
    text = f"Question: {question}\n\nAnswer: {correct_choice}{tokenizer.eos_token}"
    return {"text": text}

train_dataset = openbookqa["train"].map(format_openbookqa_train)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
print(tokenizer.decode(tokenized_train[0]))

Question: The sun is responsible for

Answer: plants sprouting, blooming and wilting<|endoftext|>


In [6]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback

training_args = TrainingArguments(
    output_dir="./mamba-1.4b-openbookqa-lora",
    per_device_train_batch_size=8,      # adjust based on GPU memory
    gradient_accumulation_steps=8,       # effective batch size = 8
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,                           # or bf16 if supported
    logging_steps=50,
    save_strategy="epoch",
    optim="paged_adamw_8bit",     
    gradient_checkpointing=True,
    report_to="none",       
    remove_unused_columns=False,
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [9]:
import time
import pynvml

pynvml.nvmlInit()
HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def sample_power_w():
    return pynvml.nvmlDeviceGetPowerUsage(HANDLE) / 1000.0

class PowerCallback(TrainerCallback):
    def __init__(self, handle, every_steps=10):
        self.handle = handle
        self.every_steps = every_steps
        self.powers = []
        self.times = []
        self.t0 = None
        self.t1 = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_steps == 0:
            self.powers.append(pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0)
            self.times.append(time.time())

    def on_train_end(self, args, state, control, **kwargs):
        self.t1 = time.time()

def summarize_training_power(cb: PowerCallback):
    duration = (cb.t1 - cb.t0) if (cb.t0 is not None and cb.t1 is not None) else None
    if not cb.powers:
        return {"train_time_sec": duration, "avg_power_w": None, "energy_j": None}

    avg_power = sum(cb.powers) / len(cb.powers)
    # energy approximation across sampled interval
    sampled_duration = (cb.times[-1] - cb.times[0]) if len(cb.times) >= 2 else duration
    energy_j = avg_power * sampled_duration if sampled_duration is not None else None

    return {"train_time_sec": duration, "avg_power_w": avg_power, "energy_j": energy_j}

In [10]:
power_cb = PowerCallback(HANDLE, every_steps=10)
trainer.add_callback(power_cb)

train_out = trainer.train()
train_metrics = summarize_training_power(power_cb)

train_out, train_metrics

Step,Training Loss
50,23.395671
100,20.709961
150,19.949060
200,18.634060


(TrainOutput(global_step=234, training_loss=20.341945289546608, metrics={'train_runtime': 14174.3422, 'train_samples_per_second': 1.049, 'train_steps_per_second': 0.017, 'total_flos': 3941862204444672.0, 'train_loss': 20.341945289546608, 'epoch': 3.0}),
 {'train_time_sec': 14174.341111421585,
  'avg_power_w': 59.579086956521735,
  'energy_j': 799201.7881232213})

In [11]:
model.save_pretrained("mamba-1.4b-openbookqa-lora-adapter")